In [0]:
#Read the CSV
source_path = "/Volumes/main/ai_quality_demo/project_files/ai_data_quality_customers_10000.csv"
df = (spark.read.option("header", True)
      .option("inferSchema", True)
      .csv(source_path))


print("Total records:", df.count())
display(df)
df.printSchema()


In [0]:
#Validate core fields
from pyspark.sql import functions as F
email_pattern = r"^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$"


validated_df = (df
    .withColumn("missing_name",
        F.col("name").isNull() | (F.trim(F.col("name")) == ""))
    .withColumn("invalid_email",
        F.col("email").isNull() | (~F.col("email").rlike(email_pattern)))
    .withColumn("invalid_age",
        F.col("age").isNull() | (~F.col("age").between(18, 100)))
    .withColumn("quality_issues",
        F.concat_ws("; ",
            F.when(F.col("missing_name"), "Missing name"),
            F.when(F.col("invalid_email"), "Invalid email"),
            F.when(F.col("invalid_age"), "Invalid age"))))


clean_df = validated_df.filter(F.col("quality_issues") == "").drop(
    "missing_name", "invalid_email", "invalid_age", "quality_issues")
bad_df = validated_df.filter(F.col("quality_issues") != "")


display(clean_df)
display(bad_df)


In [0]:
#Save Delta outputs
clean_df.write.format("delta").mode("overwrite").saveAsTable(
    "main.ai_quality_demo.clean_customers")


bad_df.write.format("delta").mode("overwrite").saveAsTable(
    "main.ai_quality_demo.rejected_customers")


In [0]:
#Build aggregate metrics
total_count = df.count()
valid_count = clean_df.count()
invalid_count = bad_df.count()


quality_summary = {
    "total_records": total_count,
    "valid_records": valid_count,
    "invalid_records": invalid_count,
    "issue_counts": {
        "missing_name": bad_df.filter(
            F.col("quality_issues").contains("Missing name")).count(),
        "invalid_email": bad_df.filter(
            F.col("quality_issues").contains("Invalid email")).count(),
        "invalid_age": bad_df.filter(
            F.col("quality_issues").contains("Invalid age")).count()
    }
}


display(quality_summary)


In [0]:
# List available secret scopes
print(dbutils.secrets.listScopes())


# List keys in the secret scope


dbutils.secrets.list("ai-quality-scope")



In [0]:
%pip install openai

In [0]:
%restart_python

In [0]:
#Calling the deployed model from microsoft foundry 
import json
from openai import OpenAI


endpoint = dbutils.secrets.get("ai-quality-scope", "azure-openai-endpoint")
api_key = dbutils.secrets.get("ai-quality-scope", "azure-openai-api-key")
deployment_name = "Phi-4-mini-instruct"


client = OpenAI(
    api_key=api_key,
    base_url=endpoint.rstrip("/").removesuffix("/chat/completions") + "/"
)


prompt = f"""
You are a data quality assistant.
Analyze these aggregate metrics. Do not invent metrics:
{json.dumps(quality_summary)}
Explain overall quality, frequent issues, business impact,
remediation suggestions and three next steps.
Do not assume invalid records should be deleted.
"""


response = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {"role": "system", "content":
         "Explain data engineering quality issues clearly and accurately."},
        {"role": "user", "content": prompt}
    ],
    temperature=1,
    max_completion_tokens=500
)
ai_report = response.choices[0].message.content


In [0]:
#Save the AI report
report_df = spark.createDataFrame(
    [(ai_report, total_count, valid_count, invalid_count)],
    ["ai_report", "total_records", "valid_records", "invalid_records"]
)


report_df.write.format("delta").mode("overwrite").saveAsTable(
    "main.ai_quality_demo.ai_quality_report")


